# 🇬🇧 UK Capital Allowance Prospecting Prioritization System

**Leyton UK Data Analytics**

---

A data-driven account prioritization system that ranks and segments UK companies into **High**, **Medium**, and **Low** priority tiers for Capital Allowance (CA) prospecting.

## 🎯 Objectives

1. Aggregate multiple CA-relevant signals at the company level
2. Normalize and engineer explainable features
3. Learn optimal signal weights from historical signed accounts
4. Generate a single priority score per company
5. Segment companies into 3 prioritization tiers
6. Maintain full explainability for each score

## 📊 Data Sources

| Source | Format | Signals | Purpose |
|--------|--------|---------|--------|
| **Input Companies** | CSV | Registration numbers | Prospecting universe |
| **Historical Accounts** | CSV | Signed/Not-signed | Weight calibration |
| **HM Land Registry** | CSV | Property ownership | Property signals |
| **DueDil/nCino** | **API** | Financials | Investment activity |

---
## 1️⃣ Setup & Configuration

In [1]:
# Core imports
import pandas as pd
import numpy as np
import json
import requests
import time
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from pathlib import Path
from tqdm.notebook import tqdm
import concurrent.futures
from functools import partial

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.style.use('seaborn-v0_8-whitegrid')

# Leyton brand colors
LEYTON_BLUE = '#012d48'
LEYTON_ORANGE = '#ec6839'
LEYTON_COLORS = [LEYTON_BLUE, LEYTON_ORANGE, '#4a90a4', '#f4a460', '#2e8b57']

print("✅ All imports successful!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

✅ All imports successful!
📅 Analysis Date: 2026-02-05 16:32


In [2]:
@dataclass
class CAPrioritizationConfig:
    """Configuration for the CA Prioritization System."""

    # Tier thresholds (percentile-based)
    high_tier_threshold: float = 0.85  # Top 15%
    medium_tier_threshold: float = 0.50  # Next 35% (50-85 percentile)

    # Default feature weights (used if insufficient calibration data)
    default_weights: Dict[str, float] = field(default_factory=lambda: {
        'tangible_score': 0.30,
        'fixed_score': 0.20,
        'ownership_score': 0.25,
        'ownership_recency_score': 0.15,
        'size_score': 0.10
    })

    # Recent acquisition threshold (years)
    recent_acquisition_years: int = 3

    # Minimum samples for calibration
    min_calibration_samples: int = 50

    # DueDil API settings
    duedil_base_url: str = "https://duedil.io/v4/company/gb"
    duedil_rate_limit_delay: float = 0.1  # seconds between requests
    duedil_max_retries: int = 3
    duedil_timeout: int = 30

# Initialize configuration
config = CAPrioritizationConfig()
print("⚙️ Configuration initialized:")
print(f"   • High Tier: Top {(1-config.high_tier_threshold)*100:.0f}%")
print(f"   • Medium Tier: {(config.high_tier_threshold-config.medium_tier_threshold)*100:.0f}%")
print(f"   • Low Tier: Bottom {config.medium_tier_threshold*100:.0f}%")
print(f"   • DueDil API: {config.duedil_base_url}")

⚙️ Configuration initialized:
   • High Tier: Top 15%
   • Medium Tier: 35%
   • Low Tier: Bottom 50%
   • DueDil API: https://duedil.io/v4/company/gb


---
## 2️⃣ DueDil API Configuration

⚠️ **Enter your DueDil API key below**

In [3]:
# =============================================================================
# 🔑 ENTER YOUR DUEDIL API KEY HERE
# =============================================================================

DUEDIL_API_KEY = "4536b65f4b990d493117fe4be989d022"  # <-- Enter your API key here

# Alternatively, use Colab secrets (recommended for security)
try:
    from google.colab import userdata
    DUEDIL_API_KEY = userdata.get('DUEDIL_API_KEY') or DUEDIL_API_KEY
    if DUEDIL_API_KEY:
        print("✅ API key loaded from Colab secrets")
except:
    pass

if not DUEDIL_API_KEY:
    print("⚠️ WARNING: No DueDil API key provided!")
    print("   Please enter your API key in the cell above.")
    print("   Or add it to Colab secrets as 'DUEDIL_API_KEY'")
else:
    print(f"✅ DueDil API key configured ({DUEDIL_API_KEY[:8]}...)")

✅ DueDil API key configured (4536b65f...)


---
## 3️⃣ Data Upload (CSV Files)

Upload your CSV files:
1. **input_companies.csv** (required) - Prospecting universe
2. **historical_accounts.csv** (optional) - For weight calibration
3. **land_registry.csv** (optional) - HM Land Registry data

In [4]:
# =============================================================================
# 📤 UPLOAD YOUR CSV FILES
# =============================================================================

USE_SAMPLE_DATA = False  # Set to True to use generated sample data instead

if not USE_SAMPLE_DATA:
    from google.colab import files

    print("📤 Please upload your CSV files:")
    print("   1. input_companies.csv (REQUIRED)")
    print("   2. historical_accounts.csv (optional - for calibration)")
    print("   3. land_registry.csv (optional - for property signals)")
    print("\n   Note: Financials will be fetched from DueDil API")
    print()

    uploaded = files.upload()
    uploaded_files = list(uploaded.keys())

    print(f"\n✅ Uploaded {len(uploaded_files)} file(s):")
    for f in uploaded_files:
        print(f"   • {f}")

📤 Please upload your CSV files:
   1. input_companies.csv (REQUIRED)
   2. historical_accounts.csv (optional - for calibration)
   3. land_registry.csv (optional - for property signals)

   Note: Financials will be fetched from DueDil API



Saving historical_accounts.csv to historical_accounts.csv
Saving input_companies.csv to input_companies.csv
Saving land_registry.csv to land_registry.csv

✅ Uploaded 3 file(s):
   • historical_accounts.csv
   • input_companies.csv
   • land_registry.csv


---
## 4️⃣ Data Parsing & Validation

In [5]:
class DataParser:
    """Handles parsing and validation of input CSV files."""

    @staticmethod
    def clean_crn(crn: str) -> str:
        """Clean and standardize company registration number."""
        crn = str(crn).strip().upper()
        crn = ''.join(c for c in crn if c.isalnum())
        # Pad numeric-only CRNs to 8 digits
        if crn.isdigit():
            crn = crn.zfill(8)
        return crn

    @staticmethod
    def parse_input_companies(filepath: str) -> pd.DataFrame:
        """Parse input companies CSV."""
        df = pd.read_csv(filepath,  sep=";",dtype=str)
        df.columns = df.columns.str.strip().str.lower()

        # Column mapping
        column_map = {
            'registered number': 'registered_number',
            'account name': 'account_name',
        }
        df = df.rename(columns=column_map)

        if 'registered_number' not in df.columns:
            raise ValueError("Input file must contain a registration number column")

        df['registered_number'] = df['registered_number'].apply(DataParser.clean_crn)
        df = df[df['registered_number'].str.len() >= 6]
        df = df.drop_duplicates(subset=['registered_number'])

        return df

    @staticmethod
    def parse_historical(filepath: str) -> pd.DataFrame:
        """Parse historical accounts CSV."""
        df = pd.read_csv(filepath,  sep=";",dtype=str)
        df.columns = df.columns.str.strip().str.lower()

        column_map = {
            'registered number': 'registered_number',
            'company registration number': 'registered_number',
            'signed or not': 'signed',
            'signed': 'signed',
            'outcome': 'signed',
            'is_signed': 'signed'
        }
        df = df.rename(columns=column_map)

        df['registered_number'] = df['registered_number'].apply(DataParser.clean_crn)
        df['signed'] = pd.to_numeric(df['signed'], errors='coerce').fillna(0).astype(int).clip(0, 1)
        df = df.drop_duplicates(subset=['registered_number'])

        return df

    @staticmethod
    def parse_land_registry(filepath: str, config: CAPrioritizationConfig) -> pd.DataFrame:
        """Parse HM Land Registry CSV and aggregate to company level."""
        df = pd.read_csv(filepath,  sep=",",dtype=str, low_memory=False)
        df.columns = df.columns.str.strip()

        # Collect proprietor records (up to 4 per title)
        proprietor_records = []
        for i in range(1, 5):
            reg_col = f'Company Registration No. ({i})'
            if reg_col in df.columns:
                subset = df[df[reg_col].notna() & (df[reg_col].astype(str).str.strip() != '')]
                if len(subset) > 0:
                    cols_to_keep = ['Title Number', 'Tenure', 'Property Address', 'Region',
                                    'Price Paid', 'Date Proprietor Added', reg_col]
                    cols_to_keep = [c for c in cols_to_keep if c in df.columns]
                    records = subset[cols_to_keep].copy()
                    records = records.rename(columns={reg_col: 'registered_number'})
                    proprietor_records.append(records)

        if not proprietor_records:
            return pd.DataFrame(columns=['registered_number'])

        long_df = pd.concat(proprietor_records, ignore_index=True)
        long_df['registered_number'] = long_df['registered_number'].apply(DataParser.clean_crn)

        # Parse dates
        if 'Date Proprietor Added' in long_df.columns:
            long_df['date_added'] = pd.to_datetime(long_df['Date Proprietor Added'], errors='coerce')
        else:
            long_df['date_added'] = pd.NaT

        cutoff_date = datetime.now() - timedelta(days=config.recent_acquisition_years * 365)

        # Aggregate to company level
        agg = long_df.groupby('registered_number').agg(
            property_count=('Title Number', 'nunique'),
            freehold_count=('Tenure', lambda x: (x.str.lower() == 'freehold').sum()),
            leasehold_count=('Tenure', lambda x: (x.str.lower() == 'leasehold').sum()),
            recent_count=('date_added', lambda x: (x >= cutoff_date).sum()),
            total_value=('Price Paid', lambda x: pd.to_numeric(x, errors='coerce').sum())
        ).reset_index()

        agg['freehold_flag'] = (agg['freehold_count'] > 0).astype(int)
        agg['leasehold_only_flag'] = ((agg['leasehold_count'] > 0) & (agg['freehold_count'] == 0)).astype(int)
        agg['recent_acquisition_flag'] = (agg['recent_count'] > 0).astype(int)

        return agg


# Parse uploaded files
parser = DataParser()

print("🔄 Parsing uploaded CSV files...")

# Input companies (required)
input_file = next((f for f in uploaded_files if 'input' in f.lower() or 'companies' in f.lower()), None)
if input_file:
    companies_df = parser.parse_input_companies(input_file)
    print(f"   ✓ Input companies: {len(companies_df):,} unique companies")
else:
    raise FileNotFoundError("input_companies.csv not found! Please upload it.")

# Historical accounts (optional)
hist_file = next((f for f in uploaded_files if 'historical' in f.lower() or 'history' in f.lower()), None)
if hist_file:
    historical_df = parser.parse_historical(hist_file)
    print(f"   ✓ Historical accounts: {len(historical_df):,} records ({historical_df['signed'].sum():,} signed)")
else:
    historical_df = None
    print("   ⚠️ No historical data - will use default weights")

# Land Registry (optional)
lr_file = next((f for f in uploaded_files if 'land' in f.lower() or 'registry' in f.lower() or 'property' in f.lower()), None)
if lr_file:
    land_registry_df = parser.parse_land_registry(lr_file, config)
    print(f"   ✓ Land Registry: {len(land_registry_df):,} companies with property")
else:
    land_registry_df = None
    print("   ⚠️ No Land Registry data - property signals unavailable")

print("\n✅ CSV parsing complete!")

🔄 Parsing uploaded CSV files...
   ✓ Input companies: 115 unique companies
   ✓ Historical accounts: 6,051 records (99 signed)
   ✓ Land Registry: 28,280 companies with property

✅ CSV parsing complete!


---
## 5️⃣ DueDil API Client & Financial Data Extraction

This section fetches financial data from the DueDil API for all companies in the prospecting universe.

In [6]:
class DueDilClient:
    """
    Client for fetching financial data from DueDil/nCino API.

    API Endpoint: GET https://duedil.io/v4/company/gb/{company_registration_number}/financials.json

    Extracted fields:
    - tangibleAssets.deltaAbsolute: YoY physical asset investment
    - fixedAssets.deltaAbsolute: YoY total fixed asset movement
    - numberOfEmployees.value: Company scale
    - turnover.value: Revenue scale
    - accountsDate: Data recency
    """

    def __init__(self, api_key: str, config: CAPrioritizationConfig):
        self.api_key = api_key
        self.config = config
        self.session = requests.Session()
        self.session.headers.update({
            'X-AUTH-TOKEN': api_key,
            'Accept': 'application/json'
        })

        # Statistics
        self.stats = {
            'success': 0,
            'not_found': 0,
            'errors': 0,
            'rate_limited': 0
        }

    def _safe_get(self, d: dict, *keys, default=None):
        """Safely navigate nested dictionary."""
        for key in keys:
            if isinstance(d, dict):
                d = d.get(key, default)
            else:
                return default
        return d

    def get_financials(self, company_number: str) -> Optional[Dict]:
        """
        Fetch financial data for a single company.

        Returns dict with extracted fields or None if not found/error.
        """
        url = f"{self.config.duedil_base_url}/{company_number}/financials.json"

        for attempt in range(self.config.duedil_max_retries):
            try:
                response = self.session.get(url, timeout=self.config.duedil_timeout)

                if response.status_code == 200:
                    data = response.json()
                    self.stats['success'] += 1
                    return self._extract_financials(data, company_number)

                elif response.status_code == 404:
                    self.stats['not_found'] += 1
                    return None

                elif response.status_code == 429:  # Rate limited
                    self.stats['rate_limited'] += 1
                    time.sleep(2 ** attempt)  # Exponential backoff
                    continue

                else:
                    self.stats['errors'] += 1
                    return None

            except requests.exceptions.Timeout:
                if attempt < self.config.duedil_max_retries - 1:
                    time.sleep(1)
                    continue
                self.stats['errors'] += 1
                return None

            except Exception as e:
                self.stats['errors'] += 1
                return None

        return None

    def _extract_financials(self, data: Dict, company_number: str) -> Dict:
        """Extract relevant fields from DueDil API response."""
        financials = data.get('financials', [])
        latest = financials[0] if financials else {}

        return {
            'registered_number': company_number,
            'tangible_assets_delta': self._safe_get(latest, 'tangibleAssets', 'deltaAbsolute'),
            'tangible_assets_value': self._safe_get(latest, 'tangibleAssets', 'value'),
            'fixed_assets_delta': self._safe_get(latest, 'fixedAssets', 'deltaAbsolute'),
            'fixed_assets_value': self._safe_get(latest, 'fixedAssets', 'value'),
            'number_of_employees': self._safe_get(latest, 'numberOfEmployees', 'value'),
            'turnover': self._safe_get(latest, 'turnover', 'value'),
            'accounts_date': self._safe_get(latest, 'accountsDate'),
            'net_assets': self._safe_get(latest, 'netAssets', 'value'),
            'total_assets': self._safe_get(latest, 'totalAssets', 'value'),
        }

    def fetch_batch(self, company_numbers: List[str],
                    show_progress: bool = True) -> pd.DataFrame:
        """
        Fetch financials for multiple companies with progress bar.
        """
        results = []

        iterator = tqdm(company_numbers, desc="Fetching financials") if show_progress else company_numbers

        for crn in iterator:
            financials = self.get_financials(crn)
            if financials:
                results.append(financials)

            # Rate limiting
            time.sleep(self.config.duedil_rate_limit_delay)

        if results:
            return pd.DataFrame(results)
        else:
            return pd.DataFrame(columns=[
                'registered_number', 'tangible_assets_delta', 'fixed_assets_delta',
                'number_of_employees', 'turnover', 'accounts_date'
            ])

    def print_stats(self):
        """Print API call statistics."""
        total = sum(self.stats.values())
        print(f"\n📊 DueDil API Statistics:")
        print(f"   • Total requests: {total:,}")
        print(f"   • Successful: {self.stats['success']:,} ({self.stats['success']/total*100:.1f}%)")
        print(f"   • Not found: {self.stats['not_found']:,}")
        print(f"   • Errors: {self.stats['errors']:,}")
        print(f"   • Rate limited: {self.stats['rate_limited']:,}")


print("✅ DueDil API client ready")

✅ DueDil API client ready


In [7]:
# =============================================================================
# 🌐 FETCH FINANCIAL DATA FROM DUEDIL API
# =============================================================================

if not DUEDIL_API_KEY:
    print("❌ Cannot fetch financials - no API key provided!")
    print("   Please enter your DueDil API key in Section 2.")
    financials_df = pd.DataFrame(columns=['registered_number'])
else:
    # Initialize client
    duedil_client = DueDilClient(DUEDIL_API_KEY, config)

    # Get list of company numbers to fetch
    company_numbers = companies_df['registered_number'].tolist()

    print(f"🌐 Fetching financial data from DueDil API...")
    print(f"   • Companies to fetch: {len(company_numbers):,}")
    print(f"   • Estimated time: ~{len(company_numbers) * 0.15 / 60:.1f} minutes")
    print()

    # Fetch financials
    financials_df = duedil_client.fetch_batch(company_numbers, show_progress=True)

    # Print statistics
    duedil_client.print_stats()

    print(f"\n✅ Financial data retrieved for {len(financials_df):,} companies")

🌐 Fetching financial data from DueDil API...
   • Companies to fetch: 115
   • Estimated time: ~0.3 minutes



Fetching financials:   0%|          | 0/115 [00:00<?, ?it/s]


📊 DueDil API Statistics:
   • Total requests: 115
   • Successful: 115 (100.0%)
   • Not found: 0
   • Errors: 0
   • Rate limited: 0

✅ Financial data retrieved for 115 companies


In [8]:
# Preview financial data
if len(financials_df) > 0:
    print("📄 Financial Data Preview:")
    display(financials_df.head(10))

    print("\n📊 Financial Data Statistics:")
    numeric_cols = ['tangible_assets_delta', 'fixed_assets_delta', 'number_of_employees', 'turnover']
    numeric_cols = [c for c in numeric_cols if c in financials_df.columns]
    display(financials_df[numeric_cols].describe())
else:
    print("⚠️ No financial data available")

📄 Financial Data Preview:


,registered_number,tangible_assets_delta,tangible_assets_value,fixed_assets_delta,fixed_assets_value,number_of_employees,turnover,accounts_date,net_assets,total_assets
0,00006775,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN
1,12470774,0.0,4396808.0,0.0,4396808.0,2.0,NaN,2025-06-30,4627503.0,4687995.0
2,SC710356,NaN,NaN,NaN,NaN,NaN,NaN,2024-06-30,-373683.0,221482.0
3,13060744,-10295.0,1019205.0,-10295.0,1019205.0,NaN,NaN,2024-06-30,-1170794.0,2250309.0
4,08461555,-736000.0,5702000.0,6488000.0,24936000.0,491.0,144388000.0,2024-12-31,49525000.0,84892000.0
5,03535589,-1432306.0,22639245.0,-1807806.0,22781745.0,124.0,6586645.0,2025-03-31,9759545.0,23765464.0
6,13038066,-8537.0,2212412.0,-8537.0,2212412.0,NaN,NaN,2024-04-30,-656223.0,2596044.0
7,SC454474,2147743.0,4271879.0,2156092.0,4296462.0,17.0,NaN,2024-12-31,2854326.0,5770673.0
8,00698452,NaN,NaN,NaN,NaN,NaN,NaN,2008-09-30,NaN,NaN
9,14006439,840302.0,11000000.0,840302.0,11000000.0,4.0,NaN,2024-06-30,1366876.0,11904153.0



📊 Financial Data Statistics:


,tangible_assets_delta,fixed_assets_delta,number_of_employees,turnover
count,9.500000e+01,9.500000e+01,82.000000,3.100000e+01
mean,5.062667e+06,5.365013e+06,123.268293,5.802730e+07
std,4.646468e+07,4.648955e+07,241.599559,6.821559e+07
min,-5.221313e+06,-3.349187e+06,1.000000,7.125000e+04
25%,-1.771500e+04,-7.646500e+03,7.250000,1.299289e+07
50%,5.601200e+04,2.225300e+05,36.500000,3.371885e+07
75%,6.246365e+05,8.480785e+05,100.750000,6.627687e+07
max,4.529077e+08,4.534106e+08,1302.000000,3.028309e+08


---
## 6️⃣ Data Merging & Feature Engineering

In [9]:
# Merge all data sources
print("🔄 Merging data sources...")

merged_df = companies_df.copy()

# Merge financials (from DueDil API)
if len(financials_df) > 0:
    merged_df = merged_df.merge(financials_df, on='registered_number', how='left')
    match_count = merged_df['tangible_assets_delta'].notna().sum()
    print(f"   ✓ Financial data: {match_count:,} matches ({match_count/len(merged_df)*100:.1f}%)")
else:
    print("   ⚠️ No financial data to merge")

# Merge Land Registry (from CSV)
if land_registry_df is not None and len(land_registry_df) > 0:
    merged_df = merged_df.merge(land_registry_df, on='registered_number', how='left')
    match_count = merged_df['property_count'].notna().sum()
    print(f"   ✓ Land Registry: {match_count:,} matches ({match_count/len(merged_df)*100:.1f}%)")
else:
    print("   ⚠️ No Land Registry data to merge")

print(f"\n📊 Merged dataset: {len(merged_df):,} companies")

🔄 Merging data sources...
   ✓ Financial data: 95 matches (82.6%)
   ✓ Land Registry: 6 matches (5.2%)

📊 Merged dataset: 115 companies


In [10]:
class FeatureEngineer:
    """Handles feature engineering and percentile normalization."""

    def __init__(self, config: CAPrioritizationConfig):
        self.config = config
        self.feature_cols = [
            'tangible_score',
            'fixed_score',
            'ownership_score',
            'ownership_recency_score',
            'size_score'
        ]

    def compute_percentile_scores(self, df: pd.DataFrame) -> pd.DataFrame:
        """Convert raw features to percentile ranks (0-1 scale)."""
        df = df.copy()

        # Financial percentile scores
        if 'tangible_assets_delta' in df.columns and df['tangible_assets_delta'].notna().sum() > 0:
            df['tangible_score'] = df['tangible_assets_delta'].rank(pct=True, na_option='bottom')
        else:
            df['tangible_score'] = 0.5

        if 'fixed_assets_delta' in df.columns and df['fixed_assets_delta'].notna().sum() > 0:
            df['fixed_score'] = df['fixed_assets_delta'].rank(pct=True, na_option='bottom')
        else:
            df['fixed_score'] = 0.5

        # Size score
        if 'number_of_employees' in df.columns and df['number_of_employees'].notna().sum() > 0:
            emp_pct = df['number_of_employees'].rank(pct=True, na_option='bottom')
        else:
            emp_pct = 0.5

        if 'turnover' in df.columns and df['turnover'].notna().sum() > 0:
            turn_pct = df['turnover'].rank(pct=True, na_option='bottom')
        else:
            turn_pct = 0.5

        df['size_score'] = (emp_pct + turn_pct) / 2

        # Ownership score
        if 'property_count' in df.columns and df['property_count'].notna().sum() > 0:
            prop_pct = df['property_count'].fillna(0).rank(pct=True, na_option='bottom')
            freehold_bonus = df.get('freehold_flag', pd.Series([0]*len(df))).fillna(0) * 0.2
            df['ownership_score'] = (prop_pct + freehold_bonus).clip(0, 1)
        else:
            df['ownership_score'] = 0.5

        # Ownership recency
        if 'recent_acquisition_flag' in df.columns:
            df['ownership_recency_score'] = df['recent_acquisition_flag'].fillna(0)
        else:
            df['ownership_recency_score'] = 0.5

        # Fill remaining NaN with neutral value
        for col in self.feature_cols:
            df[col] = df[col].fillna(0.5)

        return df


# Apply feature engineering
feature_engineer = FeatureEngineer(config)
featured_df = feature_engineer.compute_percentile_scores(merged_df)

print("✅ Feature Engineering Complete!")
print("\n📊 Feature Statistics:")
display(featured_df[feature_engineer.feature_cols].describe().round(3))

✅ Feature Engineering Complete!

📊 Feature Statistics:


,tangible_score,fixed_score,ownership_score,ownership_recency_score,size_score
count,115.000,115.000,115.000,115.000,115.000
mean,0.504,0.504,0.505,0.043,0.504
std,0.289,0.289,0.116,0.205,0.184
min,0.009,0.009,0.478,0.000,0.043
25%,0.257,0.257,0.478,0.000,0.371
50%,0.504,0.504,0.478,0.000,0.472
75%,0.752,0.752,0.478,0.000,0.750
max,0.917,0.917,1.000,1.000,0.750


In [11]:
# Visualize feature distributions
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=feature_engineer.feature_cols + [''],
    specs=[[{"type": "histogram"}]*3, [{"type": "histogram"}]*3]
)

colors = [LEYTON_BLUE, LEYTON_ORANGE, '#4a90a4', '#2e8b57', '#9370db']

for i, col in enumerate(feature_engineer.feature_cols):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    fig.add_trace(
        go.Histogram(
            x=featured_df[col],
            nbinsx=30,
            marker_color=colors[i],
            name=col
        ),
        row=row, col=col_idx
    )

fig.update_layout(
    title_text="<b>Feature Score Distributions (Percentile Normalized)</b>",
    showlegend=False,
    height=500
)

fig.show()

---
## 7️⃣ Weight Calibration (Logistic Regression)

In [12]:
class WeightCalibrator:
    """Calibrates feature weights using historical signed accounts."""

    def __init__(self, config: CAPrioritizationConfig):
        self.config = config
        self.model = None
        self.scaler = None
        self.coefficients = None
        self.weights = None
        self.cv_scores = None

    def calibrate(self, df: pd.DataFrame, historical: Optional[pd.DataFrame],
                  feature_cols: List[str]) -> Dict[str, float]:
        """Calibrate weights using logistic regression."""

        if historical is None or len(historical) == 0:
            print("⚠️ No historical data provided. Using default weights.")
            self.weights = self.config.default_weights
            return self.weights

        # Merge features with historical outcomes
        calib_df = df.merge(
            historical[['registered_number', 'signed']],
            on='registered_number',
            how='inner'
        )

        print(f"📊 Calibration data: {len(calib_df):,} companies")
        print(f"   • Signed: {calib_df['signed'].sum():,} ({calib_df['signed'].mean()*100:.1f}%)")
        print(f"   • Not Signed: {len(calib_df) - calib_df['signed'].sum():,}")

        if len(calib_df) < self.config.min_calibration_samples:
            print(f"\n⚠️ Insufficient data ({len(calib_df)} < {self.config.min_calibration_samples}). Using default weights.")
            self.weights = self.config.default_weights
            return self.weights

        # Prepare features and target
        X = calib_df[feature_cols].values
        y = calib_df['signed'].values

        # Standardize
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)

        # Fit logistic regression
        self.model = LogisticRegression(
            penalty='l2',
            C=1.0,
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        )
        self.model.fit(X_scaled, y)

        # Cross-validation
        self.cv_scores = cross_val_score(self.model, X_scaled, y, cv=5, scoring='roc_auc')

        # Extract coefficients
        self.coefficients = dict(zip(feature_cols, self.model.coef_[0]))

        # Normalize to weights
        abs_coeffs = {k: abs(v) for k, v in self.coefficients.items()}
        total = sum(abs_coeffs.values())
        self.weights = {k: v / total for k, v in abs_coeffs.items()} if total > 0 else self.config.default_weights

        print(f"\n✅ Calibration Complete!")
        print(f"   • Cross-validation AUC: {self.cv_scores.mean():.3f} (±{self.cv_scores.std():.3f})")

        return self.weights

    def get_report(self) -> Dict[str, Any]:
        """Get calibration report."""
        return {
            'status': 'calibrated' if self.model else 'default',
            'coefficients': self.coefficients,
            'weights': self.weights,
            'cv_auc_mean': float(self.cv_scores.mean()) if self.cv_scores is not None else None,
            'cv_auc_std': float(self.cv_scores.std()) if self.cv_scores is not None else None,
            'intercept': float(self.model.intercept_[0]) if self.model else None
        }


# Calibrate weights
calibrator = WeightCalibrator(config)
weights = calibrator.calibrate(featured_df, historical_df, feature_engineer.feature_cols)

print("\n⚖️ Calibrated Weights:")
for feature, weight in sorted(weights.items(), key=lambda x: -x[1]):
    bar = '█' * int(weight * 40)
    print(f"   {feature:25s}: {weight:.3f} {bar}")

📊 Calibration data: 115 companies
   • Signed: 98 (85.2%)
   • Not Signed: 17

✅ Calibration Complete!
   • Cross-validation AUC: 0.605 (±0.117)

⚖️ Calibrated Weights:
   fixed_score              : 0.337 █████████████
   ownership_score          : 0.254 ██████████
   ownership_recency_score  : 0.182 ███████
   size_score               : 0.145 █████
   tangible_score           : 0.082 ███


In [13]:
# Visualize weights
fig = go.Figure()

weight_items = sorted(weights.items(), key=lambda x: -x[1])
fig.add_trace(go.Bar(
    x=[w[1] for w in weight_items],
    y=[w[0] for w in weight_items],
    orientation='h',
    marker_color=LEYTON_BLUE,
    text=[f"{w[1]*100:.1f}%" for w in weight_items],
    textposition='outside'
))

fig.update_layout(
    title="<b>Calibrated Feature Weights</b>",
    xaxis_title="Weight",
    height=350,
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

---
## 8️⃣ Priority Scoring & Tier Assignment

In [14]:
class PriorityScorer:
    """Computes priority scores and assigns tiers."""

    def __init__(self, config: CAPrioritizationConfig, weights: Dict[str, float]):
        self.config = config
        self.weights = weights

    def compute_scores(self, df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
        """Compute weighted priority scores."""
        df = df.copy()

        df['priority_score'] = 0.0
        for col in feature_cols:
            weight = self.weights.get(col, 0)
            df['priority_score'] += df[col].fillna(0.5) * weight

        return df

    def assign_tiers(self, df: pd.DataFrame) -> pd.DataFrame:
        """Assign priority tiers based on percentile thresholds."""
        df = df.copy()

        df['score_percentile'] = df['priority_score'].rank(pct=True)

        def get_tier(pct):
            if pct >= self.config.high_tier_threshold:
                return 'High'
            elif pct >= self.config.medium_tier_threshold:
                return 'Medium'
            else:
                return 'Low'

        df['priority_tier'] = df['score_percentile'].apply(get_tier)

        return df

    def add_explainability(self, df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
        """Add top drivers for explainability."""
        df = df.copy()

        def get_top_drivers(row):
            contributions = []
            for col in feature_cols:
                value = row.get(col, 0.5)
                weight = self.weights.get(col, 0)
                contribution = value * weight
                contributions.append((col, value, contribution))

            contributions.sort(key=lambda x: -x[2])
            top_3 = contributions[:3]
            return "; ".join([f"{c[0]}={c[1]:.2f}" for c in top_3])

        df['top_drivers'] = df.apply(get_top_drivers, axis=1)

        return df


# Score all companies
scorer = PriorityScorer(config, weights)

print("🔄 Computing priority scores...")
scored_df = scorer.compute_scores(featured_df, feature_engineer.feature_cols)

print("🔄 Assigning priority tiers...")
scored_df = scorer.assign_tiers(scored_df)

print("🔄 Adding explainability...")
scored_df = scorer.add_explainability(scored_df, feature_engineer.feature_cols)

print("\n✅ Scoring Complete!")

🔄 Computing priority scores...
🔄 Assigning priority tiers...
🔄 Adding explainability...

✅ Scoring Complete!


In [15]:
# Display tier distribution
tier_counts = scored_df['priority_tier'].value_counts()

print("📊 PRIORITY TIER DISTRIBUTION")
print("=" * 50)
for tier in ['High', 'Medium', 'Low']:
    count = tier_counts.get(tier, 0)
    pct = count / len(scored_df) * 100
    bar = '█' * int(pct / 2)
    print(f"   {tier:8s}: {count:5,} ({pct:5.1f}%) {bar}")

print(f"\n   Total:    {len(scored_df):5,} companies scored")

📊 PRIORITY TIER DISTRIBUTION
   High    :    22 ( 19.1%) █████████
   Medium  :    36 ( 31.3%) ███████████████
   Low     :    57 ( 49.6%) ████████████████████████

   Total:      115 companies scored


In [16]:
# Tier visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Tier Distribution', 'Score Distribution by Tier'),
    specs=[[{"type": "pie"}, {"type": "box"}]]
)

# Pie chart
fig.add_trace(
    go.Pie(
        values=[tier_counts.get('High', 0), tier_counts.get('Medium', 0), tier_counts.get('Low', 0)],
        labels=['High', 'Medium', 'Low'],
        marker_colors=[LEYTON_ORANGE, LEYTON_BLUE, '#cccccc'],
        hole=0.4,
        textinfo='label+percent'
    ),
    row=1, col=1
)

# Box plot
for tier, color in [('High', LEYTON_ORANGE), ('Medium', LEYTON_BLUE), ('Low', '#888888')]:
    tier_data = scored_df[scored_df['priority_tier'] == tier]['priority_score']
    fig.add_trace(
        go.Box(y=tier_data, name=tier, marker_color=color),
        row=1, col=2
    )

fig.update_layout(
    title_text="<b>Priority Tier Analysis</b>",
    height=400,
    showlegend=False
)

fig.show()

---
## 9️⃣ Results Analysis & Validation

In [17]:
# Top scoring companies
print("🏆 TOP 15 HIGH-PRIORITY COMPANIES")
print("=" * 80)

top_15 = scored_df.nlargest(15, 'priority_score')[[
    'registered_number', 'account_name', 'priority_score',
    'priority_tier', 'top_drivers'
]]

for i, (_, row) in enumerate(top_15.iterrows(), 1):
    name = row.get('account_name', 'N/A')
    print(f"\n{i}. {name} ({row['registered_number']})")
    print(f"   Score: {row['priority_score']:.3f} | Tier: {row['priority_tier']}")
    print(f"   Drivers: {row['top_drivers']}")

🏆 TOP 15 HIGH-PRIORITY COMPANIES

1. Xeal Investments Limited (15851751)
   Score: 0.929 | Tier: High
   Drivers: fixed_score=0.92; ownership_score=1.00; ownership_recency_score=1.00

2. Compass Community Schools Limited (15250171)
   Score: 0.911 | Tier: High
   Drivers: fixed_score=0.92; ownership_score=0.99; ownership_recency_score=1.00

3. 360 Global Properties Ltd (04469754)
   Score: 0.655 | Tier: High
   Drivers: ownership_score=1.00; ownership_recency_score=1.00; size_score=0.75

4. Paymán Holdings 7 Ltd (13060744)
   Score: 0.632 | Tier: High
   Drivers: ownership_score=1.00; ownership_recency_score=1.00; size_score=0.75

5. Raimes, Clark & Company, Limited (00006775)
   Score: 0.615 | Tier: High
   Drivers: fixed_score=0.92; ownership_score=0.48; size_score=0.75

6. Paymán Holdings 11 Ltd (SC710356)
   Score: 0.615 | Tier: High
   Drivers: fixed_score=0.92; ownership_score=0.48; size_score=0.75

7. Ardersier Port (Scotland) Limited (00698452)
   Score: 0.615 | Tier: High
   D

In [18]:
# Validation: Check if high tier has higher signed rate
if historical_df is not None and len(historical_df) > 0:
    validation_df = scored_df.merge(
        historical_df[['registered_number', 'signed']],
        on='registered_number',
        how='inner'
    )

    if len(validation_df) > 0:
        print("✅ VALIDATION: Signed Rate by Priority Tier")
        print("=" * 50)

        for tier in ['High', 'Medium', 'Low']:
            tier_data = validation_df[validation_df['priority_tier'] == tier]
            if len(tier_data) > 0:
                signed_rate = tier_data['signed'].mean() * 100
                signed_count = tier_data['signed'].sum()
                total = len(tier_data)
                bar = '█' * int(signed_rate)
                print(f"   {tier:8s}: {signed_rate:5.1f}% signed ({signed_count}/{total}) {bar}")

        # Calculate lift
        overall_rate = validation_df['signed'].mean() * 100
        high_tier_data = validation_df[validation_df['priority_tier'] == 'High']
        high_rate = high_tier_data['signed'].mean() * 100 if len(high_tier_data) > 0 else 0
        lift = high_rate / overall_rate if overall_rate > 0 else 0

        print(f"\n   📈 Overall signed rate: {overall_rate:.1f}%")
        print(f"   📈 High tier lift: {lift:.2f}x")

        # Visualization
        tier_validation = validation_df.groupby('priority_tier').agg(
            signed_rate=('signed', 'mean'),
            total=('signed', 'count')
        ).reset_index()
        tier_validation['signed_rate'] *= 100

        tier_order = {'High': 0, 'Medium': 1, 'Low': 2}
        tier_validation['order'] = tier_validation['priority_tier'].map(tier_order)
        tier_validation = tier_validation.sort_values('order')

        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=tier_validation['priority_tier'],
            y=tier_validation['signed_rate'],
            marker_color=[LEYTON_ORANGE, LEYTON_BLUE, '#cccccc'],
            text=[f"{r:.1f}%" for r in tier_validation['signed_rate']],
            textposition='outside'
        ))
        fig.add_hline(y=overall_rate, line_dash="dash", line_color="red",
                      annotation_text=f"Overall: {overall_rate:.1f}%")
        fig.update_layout(
            title="<b>Signed Rate by Priority Tier (Validation)</b>",
            xaxis_title="Priority Tier",
            yaxis_title="Signed Rate (%)",
            height=400
        )
        fig.show()
else:
    print("⚠️ No historical data available for validation")

✅ VALIDATION: Signed Rate by Priority Tier
   High    :  86.4% signed (19/22) ██████████████████████████████████████████████████████████████████████████████████████
   Medium  :  77.8% signed (28/36) █████████████████████████████████████████████████████████████████████████████
   Low     :  89.5% signed (51/57) █████████████████████████████████████████████████████████████████████████████████████████

   📈 Overall signed rate: 85.2%
   📈 High tier lift: 1.01x


---
## 📊 Feature Analysis & Model Validation Visualizations

This section provides detailed visual analysis of the scoring model's features and validation metrics, comparing signed vs. non-signed accounts across all key dimensions.

In [19]:
# =============================================================================
# 1️⃣ Distribution of Tangible Asset Investment Scores
# Purpose: Demonstrate that signed accounts tend to show higher recent physical investment activity.
# =============================================================================

if historical_df is not None and len(historical_df) > 0:
    # Merge scored data with signed labels
    analysis_df = scored_df.merge(
        historical_df[['registered_number', 'signed']],
        on='registered_number',
        how='inner'
    )

    fig = go.Figure()

    # Signed accounts
    signed_data = analysis_df[analysis_df['signed'] == 1]['tangible_score'].dropna()
    fig.add_trace(go.Histogram(
        x=signed_data,
        name='Signed',
        opacity=0.7,
        marker_color=LEYTON_ORANGE,
        nbinsx=20
    ))

    # Not signed accounts
    not_signed_data = analysis_df[analysis_df['signed'] == 0]['tangible_score'].dropna()
    fig.add_trace(go.Histogram(
        x=not_signed_data,
        name='Not Signed',
        opacity=0.7,
        marker_color=LEYTON_BLUE,
        nbinsx=20
    ))

    fig.update_layout(
        title='<b>1️⃣ Distribution of Tangible Asset Investment Scores</b>',
        xaxis_title='Tangible Score (Percentile)',
        yaxis_title='Count',
        barmode='overlay',
        legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99),
        height=400
    )
    fig.show()

    print(f"\n📊 Insight: Signed median={signed_data.median():.3f}, Not Signed median={not_signed_data.median():.3f}")
else:
    print('⚠️ Historical data required for this visualization')


📊 Insight: Signed median=0.483, Not Signed median=0.678


In [20]:
# =============================================================================
# 2️⃣ Distribution of Fixed Asset Movement Scores
# Purpose: Validate fixed asset YoY movement as a supporting Capital Allowance signal.
# =============================================================================

if historical_df is not None and len(historical_df) > 0:
    fig = go.Figure()

    # Signed accounts
    signed_data = analysis_df[analysis_df['signed'] == 1]['fixed_score'].dropna()
    fig.add_trace(go.Histogram(
        x=signed_data,
        name='Signed',
        opacity=0.7,
        marker_color=LEYTON_ORANGE,
        nbinsx=20
    ))

    # Not signed accounts
    not_signed_data = analysis_df[analysis_df['signed'] == 0]['fixed_score'].dropna()
    fig.add_trace(go.Histogram(
        x=not_signed_data,
        name='Not Signed',
        opacity=0.7,
        marker_color=LEYTON_BLUE,
        nbinsx=20
    ))

    fig.update_layout(
        title='<b>2️⃣ Distribution of Fixed Asset Movement Scores</b>',
        xaxis_title='Fixed Score (Percentile)',
        yaxis_title='Count',
        barmode='overlay',
        legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99),
        height=400
    )
    fig.show()

    print(f"\n📊 Insight: Signed median={signed_data.median():.3f}, Not Signed median={not_signed_data.median():.3f}")
    print("   Expected: Weaker separation than tangible assets (secondary signal)")
else:
    print('⚠️ Historical data required for this visualization')


📊 Insight: Signed median=0.483, Not Signed median=0.687
   Expected: Weaker separation than tangible assets (secondary signal)


In [21]:
# =============================================================================
# 3️⃣ Tangible Investment Score — Boxplot by Outcome
# Purpose: Highlight median and variance differences between signed and non-signed accounts.
# =============================================================================

if historical_df is not None and len(historical_df) > 0:
    fig = go.Figure()

    fig.add_trace(go.Box(
        y=analysis_df[analysis_df['signed'] == 0]['tangible_score'],
        name='Not Signed',
        marker_color=LEYTON_BLUE,
        boxpoints='outliers'
    ))

    fig.add_trace(go.Box(
        y=analysis_df[analysis_df['signed'] == 1]['tangible_score'],
        name='Signed',
        marker_color=LEYTON_ORANGE,
        boxpoints='outliers'
    ))

    fig.update_layout(
        title='<b>3️⃣ Tangible Investment Score — Boxplot by Outcome</b>',
        yaxis_title='Tangible Score (Percentile)',
        height=400,
        showlegend=True
    )
    fig.show()

    # Statistical summary
    signed_stats = analysis_df[analysis_df['signed'] == 1]['tangible_score'].describe()
    not_signed_stats = analysis_df[analysis_df['signed'] == 0]['tangible_score'].describe()
    print(f"\n📊 Insight: Signed accounts show higher medians ({signed_stats['50%']:.3f} vs {not_signed_stats['50%']:.3f})")
else:
    print('⚠️ Historical data required for this visualization')


📊 Insight: Signed accounts show higher medians (0.483 vs 0.678)


In [22]:
# =============================================================================
# 4️⃣ Fixed Asset Score — Boxplot by Outcome
# Purpose: Assess consistency of fixed asset signal across outcomes.
# =============================================================================

if historical_df is not None and len(historical_df) > 0:
    fig = go.Figure()

    fig.add_trace(go.Box(
        y=analysis_df[analysis_df['signed'] == 0]['fixed_score'],
        name='Not Signed',
        marker_color=LEYTON_BLUE,
        boxpoints='outliers'
    ))

    fig.add_trace(go.Box(
        y=analysis_df[analysis_df['signed'] == 1]['fixed_score'],
        name='Signed',
        marker_color=LEYTON_ORANGE,
        boxpoints='outliers'
    ))

    fig.update_layout(
        title='<b>4️⃣ Fixed Asset Score — Boxplot by Outcome</b>',
        yaxis_title='Fixed Score (Percentile)',
        height=400,
        showlegend=True
    )
    fig.show()

    # Statistical summary
    signed_stats = analysis_df[analysis_df['signed'] == 1]['fixed_score'].describe()
    not_signed_stats = analysis_df[analysis_df['signed'] == 0]['fixed_score'].describe()
    print(f"\n📊 Insight: Fixed score median - Signed: {signed_stats['50%']:.3f}, Not Signed: {not_signed_stats['50%']:.3f}")
else:
    print('⚠️ Historical data required for this visualization')


📊 Insight: Fixed score median - Signed: 0.483, Not Signed: 0.687


In [23]:
# =============================================================================
# 5️⃣ Freehold Ownership Rate by Outcome
# Purpose: Demonstrate the importance of property ownership type in CA targeting.
# =============================================================================

if historical_df is not None and len(historical_df) > 0 and 'freehold_flag' in analysis_df.columns:
    # Calculate freehold rates by signed status
    freehold_rates = analysis_df.groupby('signed')['freehold_flag'].mean() * 100

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=['Not Signed', 'Signed'],
        y=[freehold_rates.get(0, 0), freehold_rates.get(1, 0)],
        marker_color=[LEYTON_BLUE, LEYTON_ORANGE],
        text=[f'{freehold_rates.get(0, 0):.1f}%', f'{freehold_rates.get(1, 0):.1f}%'],
        textposition='outside'
    ))

    fig.update_layout(
        title='<b>5️⃣ Freehold Ownership Rate by Outcome</b>',
        xaxis_title='Outcome',
        yaxis_title='Freehold Ownership Rate (%)',
        height=400,
        yaxis=dict(range=[0, max(freehold_rates.max() * 1.3, 100)])
    )
    fig.show()

    print(f"\n📊 Insight: Signed accounts are {'more' if freehold_rates.get(1, 0) > freehold_rates.get(0, 0) else 'less'} likely to own freehold property")
else:
    print('⚠️ Historical data and Land Registry data required for this visualization')


📊 Insight: Signed accounts are less likely to own freehold property


In [24]:
# =============================================================================
# 6️⃣ Property Ownership Profile by Outcome
# Purpose: Compare tenure structures between signed and non-signed accounts.
# =============================================================================

if historical_df is not None and len(historical_df) > 0:
    # Calculate property ownership profile
    has_freehold = 'freehold_flag' in analysis_df.columns
    has_leasehold = 'leasehold_only_flag' in analysis_df.columns

    if has_freehold or has_leasehold:
        profile_data = []
        for signed_val, label in [(0, 'Not Signed'), (1, 'Signed')]:
            subset = analysis_df[analysis_df['signed'] == signed_val]
            total = len(subset)
            if total > 0:
                freehold_pct = (subset['freehold_flag'].sum() / total * 100) if has_freehold else 0
                leasehold_pct = (subset['leasehold_only_flag'].sum() / total * 100) if has_leasehold else 0
                other_pct = 100 - freehold_pct - leasehold_pct
                profile_data.append({'label': label, 'Freehold': freehold_pct, 'Leasehold Only': leasehold_pct, 'Other/Unknown': other_pct})

        if profile_data:
            fig = go.Figure()

            categories = ['Freehold', 'Leasehold Only', 'Other/Unknown']
            colors = [LEYTON_ORANGE, LEYTON_BLUE, '#cccccc']

            for cat, color in zip(categories, colors):
                fig.add_trace(go.Bar(
                    name=cat,
                    x=[d['label'] for d in profile_data],
                    y=[d[cat] for d in profile_data],
                    marker_color=color,
                    text=[f'{d[cat]:.1f}%' for d in profile_data],
                    textposition='inside'
                ))

            fig.update_layout(
                title='<b>6️⃣ Property Ownership Profile by Outcome</b>',
                xaxis_title='Outcome',
                yaxis_title='Percentage (%)',
                barmode='group',
                height=400,
                legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99)
            )
            fig.show()
    else:
        print('⚠️ Property ownership columns not available')
else:
    print('⚠️ Historical data required for this visualization')

In [25]:
# =============================================================================
# 7️⃣ Recent Property Acquisition Rate by Outcome
# Purpose: Validate recency of property acquisition as a reinforcing CA signal.
# =============================================================================

if historical_df is not None and len(historical_df) > 0 and 'recent_acquisition_flag' in analysis_df.columns:
    # Calculate recent acquisition rates by signed status
    recent_rates = analysis_df.groupby('signed')['recent_acquisition_flag'].mean() * 100

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=['Not Signed', 'Signed'],
        y=[recent_rates.get(0, 0), recent_rates.get(1, 0)],
        marker_color=[LEYTON_BLUE, LEYTON_ORANGE],
        text=[f'{recent_rates.get(0, 0):.1f}%', f'{recent_rates.get(1, 0):.1f}%'],
        textposition='outside'
    ))

    fig.update_layout(
        title=f'<b>7️⃣ Recent Property Acquisition Rate by Outcome (≤{config.recent_acquisition_years} years)</b>',
        xaxis_title='Outcome',
        yaxis_title='Recent Acquisition Rate (%)',
        height=400,
        yaxis=dict(range=[0, max(recent_rates.max() * 1.3, 100) if recent_rates.max() > 0 else 20])
    )
    fig.show()

    print(f"\n📊 Insight: Signed accounts are {'more' if recent_rates.get(1, 0) > recent_rates.get(0, 0) else 'less'} likely to have recent property acquisitions")
else:
    print('⚠️ Historical data and Land Registry data required for this visualization')


📊 Insight: Signed accounts are less likely to have recent property acquisitions


In [26]:
# =============================================================================
# 8️⃣ Distribution of Property Count by Outcome
# Purpose: Assess whether signed accounts tend to own more properties.
# =============================================================================

if historical_df is not None and len(historical_df) > 0 and 'property_count' in analysis_df.columns:
    fig = go.Figure()

    # Signed accounts
    signed_data = analysis_df[analysis_df['signed'] == 1]['property_count'].dropna()
    fig.add_trace(go.Histogram(
        x=signed_data,
        name='Signed',
        opacity=0.7,
        marker_color=LEYTON_ORANGE,
        nbinsx=15
    ))

    # Not signed accounts
    not_signed_data = analysis_df[analysis_df['signed'] == 0]['property_count'].dropna()
    fig.add_trace(go.Histogram(
        x=not_signed_data,
        name='Not Signed',
        opacity=0.7,
        marker_color=LEYTON_BLUE,
        nbinsx=15
    ))

    fig.update_layout(
        title='<b>8️⃣ Distribution of Property Count by Outcome</b>',
        xaxis_title='Property Count',
        yaxis_title='Count',
        barmode='overlay',
        legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99),
        height=400
    )
    fig.show()

    if len(signed_data) > 0 and len(not_signed_data) > 0:
        print(f"\n📊 Insight: Signed median properties={signed_data.median():.0f}, Not Signed median={not_signed_data.median():.0f}")
else:
    print('⚠️ Historical data and Land Registry property count data required for this visualization')

In [27]:
# =============================================================================
# 9️⃣ Employee Count Distribution by Outcome
# Purpose: Validate company scale as a normalization signal (not a primary driver).
# =============================================================================

if historical_df is not None and len(historical_df) > 0 and 'number_of_employees' in analysis_df.columns:
    fig = go.Figure()

    # Signed accounts (log scale for better visualization)
    signed_data = analysis_df[analysis_df['signed'] == 1]['number_of_employees'].dropna()
    signed_data = signed_data[signed_data > 0]  # Filter zeros for log

    fig.add_trace(go.Histogram(
        x=signed_data,
        name='Signed',
        opacity=0.7,
        marker_color=LEYTON_ORANGE,
        nbinsx=20
    ))

    # Not signed accounts
    not_signed_data = analysis_df[analysis_df['signed'] == 0]['number_of_employees'].dropna()
    not_signed_data = not_signed_data[not_signed_data > 0]

    fig.add_trace(go.Histogram(
        x=not_signed_data,
        name='Not Signed',
        opacity=0.7,
        marker_color=LEYTON_BLUE,
        nbinsx=20
    ))

    fig.update_layout(
        title='<b>9️⃣ Employee Count Distribution by Outcome</b>',
        xaxis_title='Number of Employees',
        yaxis_title='Count',
        barmode='overlay',
        legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99),
        height=400,
        xaxis_type='log'  # Log scale for employee count
    )
    fig.show()

    if len(signed_data) > 0 and len(not_signed_data) > 0:
        print(f"\n📊 Insight: Signed median employees={signed_data.median():.0f}, Not Signed median={not_signed_data.median():.0f}")
        print("   Note: Employee count is used for normalization, not as a primary driver")
else:
    print('⚠️ Historical data and employee count data required for this visualization')


📊 Insight: Signed median employees=37, Not Signed median=32
   Note: Employee count is used for normalization, not as a primary driver


In [28]:
# =============================================================================
# 🔟 Turnover Distribution by Outcome
# Purpose: Assess revenue scale differences between signed and non-signed accounts.
# =============================================================================

if historical_df is not None and len(historical_df) > 0 and 'turnover' in analysis_df.columns:
    fig = go.Figure()

    # Signed accounts
    signed_data = analysis_df[analysis_df['signed'] == 1]['turnover'].dropna()
    signed_data = signed_data[signed_data > 0] / 1e6  # Convert to millions

    fig.add_trace(go.Histogram(
        x=signed_data,
        name='Signed',
        opacity=0.7,
        marker_color=LEYTON_ORANGE,
        nbinsx=20
    ))

    # Not signed accounts
    not_signed_data = analysis_df[analysis_df['signed'] == 0]['turnover'].dropna()
    not_signed_data = not_signed_data[not_signed_data > 0] / 1e6

    fig.add_trace(go.Histogram(
        x=not_signed_data,
        name='Not Signed',
        opacity=0.7,
        marker_color=LEYTON_BLUE,
        nbinsx=20
    ))

    fig.update_layout(
        title='<b>🔟 Turnover Distribution by Outcome</b>',
        xaxis_title='Turnover (£ Millions)',
        yaxis_title='Count',
        barmode='overlay',
        legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99),
        height=400,
        xaxis_type='log'  # Log scale for turnover
    )
    fig.show()

    if len(signed_data) > 0 and len(not_signed_data) > 0:
        print(f"\n📊 Insight: Signed median turnover=£{signed_data.median():.1f}M, Not Signed median=£{not_signed_data.median():.1f}M")
else:
    print('⚠️ Historical data and turnover data required for this visualization')


📊 Insight: Signed median turnover=£32.2M, Not Signed median=£33.7M


In [29]:
# =============================================================================
# 1️⃣1️⃣ Signed Rate by Priority Tier (Model Validation)
# Purpose: Demonstrate that the prioritization model meaningfully ranks accounts.
# Expected: Clear monotonic decrease from High → Medium → Low tiers.
# =============================================================================

if historical_df is not None and len(historical_df) > 0:
    tier_validation = analysis_df.groupby('priority_tier').agg(
        signed_rate=('signed', 'mean'),
        signed_count=('signed', 'sum'),
        total=('signed', 'count')
    ).reset_index()
    tier_validation['signed_rate'] *= 100

    # Sort by tier order
    tier_order = {'High': 0, 'Medium': 1, 'Low': 2}
    tier_validation['order'] = tier_validation['priority_tier'].map(tier_order)
    tier_validation = tier_validation.sort_values('order')

    # Overall rate for reference line
    overall_rate = analysis_df['signed'].mean() * 100

    fig = go.Figure()

    colors = [LEYTON_ORANGE, LEYTON_BLUE, '#cccccc']
    fig.add_trace(go.Bar(
        x=tier_validation['priority_tier'],
        y=tier_validation['signed_rate'],
        marker_color=colors[:len(tier_validation)],
        text=[f"{r:.1f}%<br>({int(c)}/{int(t)})" for r, c, t in zip(
            tier_validation['signed_rate'],
            tier_validation['signed_count'],
            tier_validation['total']
        )],
        textposition='outside'
    ))

    # Add reference line
    fig.add_hline(y=overall_rate, line_dash='dash', line_color='red',
                  annotation_text=f'Overall: {overall_rate:.1f}%', annotation_position='right')

    fig.update_layout(
        title='<b>1️⃣1️⃣ Signed Rate by Priority Tier (Model Validation)</b>',
        xaxis_title='Priority Tier',
        yaxis_title='Signed Rate (%)',
        height=450,
        yaxis=dict(range=[0, tier_validation['signed_rate'].max() * 1.3])
    )
    fig.show()

    # Check for monotonic decrease
    rates = tier_validation.set_index('priority_tier')['signed_rate']
    is_monotonic = rates.get('High', 0) >= rates.get('Medium', 0) >= rates.get('Low', 0)
    print(f"\n📊 Validation: {'✅ Model shows expected monotonic decrease' if is_monotonic else '⚠️ Non-monotonic pattern detected'}")
    print(f"   High tier lift: {rates.get('High', 0) / overall_rate:.2f}x vs overall")
else:
    print('⚠️ Historical data required for this visualization')


📊 Validation: ⚠️ Non-monotonic pattern detected
   High tier lift: 1.01x vs overall


In [30]:
# =============================================================================
# 1️⃣2️⃣ Overall Priority Score Distribution (All Accounts)
# Purpose: Provide visibility into score spread and confirm absence of extreme clustering.
# =============================================================================

fig = go.Figure()

# Add histogram with tier coloring
for tier, color in [('High', LEYTON_ORANGE), ('Medium', LEYTON_BLUE), ('Low', '#cccccc')]:
    tier_data = scored_df[scored_df['priority_tier'] == tier]['priority_score']
    fig.add_trace(go.Histogram(
        x=tier_data,
        name=f'{tier} Tier',
        marker_color=color,
        opacity=0.8,
        nbinsx=15
    ))

# Add tier threshold lines
high_threshold = scored_df['priority_score'].quantile(config.high_tier_threshold)
medium_threshold = scored_df['priority_score'].quantile(config.medium_tier_threshold)

fig.add_vline(x=high_threshold, line_dash='dash', line_color=LEYTON_ORANGE,
              annotation_text=f'High: {high_threshold:.3f}', annotation_position='top')
fig.add_vline(x=medium_threshold, line_dash='dash', line_color=LEYTON_BLUE,
              annotation_text=f'Medium: {medium_threshold:.3f}', annotation_position='bottom')

fig.update_layout(
    title='<b>1️⃣2️⃣ Overall Priority Score Distribution (All Accounts)</b>',
    xaxis_title='Priority Score',
    yaxis_title='Count',
    barmode='stack',
    legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99),
    height=400
)
fig.show()

# Score distribution statistics
print(f"\n📊 Score Distribution:")
print(f"   Min: {scored_df['priority_score'].min():.3f}")
print(f"   25th percentile: {scored_df['priority_score'].quantile(0.25):.3f}")
print(f"   Median: {scored_df['priority_score'].median():.3f}")
print(f"   75th percentile: {scored_df['priority_score'].quantile(0.75):.3f}")
print(f"   Max: {scored_df['priority_score'].max():.3f}")
print(f"   Std Dev: {scored_df['priority_score'].std():.3f}")


📊 Score Distribution:
   Min: 0.185
   25th percentile: 0.315
   Median: 0.412
   75th percentile: 0.508
   Max: 0.929
   Std Dev: 0.144


In [31]:
# =============================================================================
# 1️⃣3️⃣ Top Drivers Contribution Breakdown (Explainability)
# Purpose: Illustrate explainability of the model with feature contributions.
# =============================================================================

# Calculate average contributions across all accounts
feature_cols = ['tangible_score', 'fixed_score', 'ownership_score', 'ownership_recency_score', 'size_score']
feature_cols = [c for c in feature_cols if c in scored_df.columns]

avg_contributions = {}
for col in feature_cols:
    avg_value = scored_df[col].fillna(0.5).mean()
    weight = weights.get(col, 0)
    contribution = avg_value * weight
    avg_contributions[col] = {
        'avg_value': avg_value,
        'weight': weight,
        'contribution': contribution
    }

# Sort by contribution
sorted_features = sorted(avg_contributions.items(), key=lambda x: -x[1]['contribution'])

fig = go.Figure()

fig.add_trace(go.Bar(
    y=[f[0].replace('_score', '').replace('_', ' ').title() for f in sorted_features],
    x=[f[1]['contribution'] for f in sorted_features],
    orientation='h',
    marker_color=LEYTON_BLUE,
    text=[f"{f[1]['contribution']:.3f} (avg={f[1]['avg_value']:.2f} × wt={f[1]['weight']:.2f})" for f in sorted_features],
    textposition='outside'
))

fig.update_layout(
    title='<b>1️⃣3️⃣ Average Feature Contribution to Priority Score</b>',
    xaxis_title='Contribution to Score',
    yaxis_title='Feature',
    yaxis=dict(categoryorder='total ascending'),
    height=350,
    margin=dict(l=150)
)
fig.show()

# Show example for top scored company
print("\n📊 Example: Top Scored Company Breakdown")
top_company = scored_df.nlargest(1, 'priority_score').iloc[0]
print(f"   Company: {top_company.get('account_name', 'N/A')}")
print(f"   Priority Score: {top_company['priority_score']:.3f}")
print(f"   Top Drivers: {top_company.get('top_drivers', 'N/A')}")

# Feature breakdown for top company
print("\n   Feature Contributions:")
for col in feature_cols:
    value = top_company.get(col, 0.5)
    weight = weights.get(col, 0)
    contrib = value * weight
    print(f"   • {col}: {value:.3f} × {weight:.3f} = {contrib:.3f}")


📊 Example: Top Scored Company Breakdown
   Company: Xeal Investments Limited
   Priority Score: 0.929
   Top Drivers: fixed_score=0.92; ownership_score=1.00; ownership_recency_score=1.00

   Feature Contributions:
   • tangible_score: 0.917 × 0.082 = 0.075
   • fixed_score: 0.917 × 0.337 = 0.309
   • ownership_score: 1.000 × 0.254 = 0.254
   • ownership_recency_score: 1.000 × 0.182 = 0.182
   • size_score: 0.750 × 0.145 = 0.109


---
## 🔟 Export Results (Salesforce-Ready)

In [32]:
# Prepare final output
export_cols = [
    'registered_number',
    'account_name',
    'priority_score',
    'priority_tier',
    'score_percentile',
    'top_drivers',
    'tangible_score',
    'fixed_score',
    'ownership_score',
    'ownership_recency_score',
    'size_score'
]

# Add raw data columns
raw_cols = [
    'tangible_assets_delta', 'fixed_assets_delta', 'tangible_assets_value',
    'number_of_employees', 'turnover', 'accounts_date',
    'property_count', 'freehold_flag', 'recent_acquisition_flag'
]

for col in raw_cols:
    if col in scored_df.columns:
        export_cols.append(col)

export_cols = [c for c in export_cols if c in scored_df.columns]
final_output = scored_df[export_cols].copy()

# Round numeric columns
numeric_cols = final_output.select_dtypes(include=[np.number]).columns
final_output[numeric_cols] = final_output[numeric_cols].round(4)

print("📄 Final Output Preview:")
display(final_output.head(10))

📄 Final Output Preview:


,registered_number,account_name,priority_score,priority_tier,score_percentile,top_drivers,tangible_score,fixed_score,ownership_score,ownership_recency_score,size_score,tangible_assets_delta,fixed_assets_delta,tangible_assets_value,number_of_employees,turnover,accounts_date,property_count,freehold_flag,recent_acquisition_flag
0,00006775,"Raimes, Clark & Company, Limited",0.6146,High,0.8913,fixed_score=0.92; ownership_score=0.48; size_s...,0.9174,0.9174,0.4783,0.0,0.7500,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
1,12470774,O.C.L Property Co Limited,0.2981,Low,0.2261,ownership_score=0.48; fixed_score=0.29; size_s...,0.3478,0.2870,0.4783,0.0,0.3543,0.0,0.0,4396808.0,2.0,NaN,2025-06-30,NaN,NaN,NaN
2,SC710356,Paymán Holdings 11 Ltd,0.6146,High,0.8913,fixed_score=0.92; ownership_score=0.48; size_s...,0.9174,0.9174,0.4783,0.0,0.7500,NaN,NaN,NaN,NaN,NaN,2024-06-30,NaN,NaN,NaN
3,13060744,Paymán Holdings 7 Ltd,0.6321,High,0.9739,ownership_score=1.00; ownership_recency_score=...,0.2435,0.2000,1.0000,1.0,0.7500,-10295.0,-10295.0,1019205.0,NaN,NaN,2024-06-30,1.0,1.0,1.0
4,08461555,Seddon Group Limited,0.4653,Medium,0.6783,fixed_score=0.81; ownership_score=0.48; size_s...,0.0783,0.8087,0.4783,0.0,0.4478,-736000.0,6488000.0,5702000.0,491.0,144388000.0,2024-12-31,NaN,NaN,NaN
5,03535589,Glen Mhor Limited,0.1846,Low,0.0087,ownership_score=0.48; size_score=0.30; fixed_s...,0.0609,0.0435,0.4783,0.0,0.3000,-1432306.0,-1807806.0,22639245.0,124.0,6586645.0,2025-03-31,NaN,NaN,NaN
6,13038066,Paymán Holdings 5 Ltd,0.3212,Low,0.2609,ownership_score=0.48; size_score=0.75; fixed_s...,0.2522,0.2087,0.4783,0.0,0.7500,-8537.0,-8537.0,2212412.0,NaN,NaN,2024-04-30,NaN,NaN,NaN
7,SC454474,Dunnet Bay Distillers LTD.,0.4992,Medium,0.7391,fixed_score=0.74; ownership_score=0.48; size_s...,0.7652,0.7391,0.4783,0.0,0.4543,2147743.0,2156092.0,4271879.0,17.0,NaN,2024-12-31,NaN,NaN,NaN
8,00698452,Ardersier Port (Scotland) Limited,0.6146,High,0.8913,fixed_score=0.92; ownership_score=0.48; size_s...,0.9174,0.9174,0.4783,0.0,0.7500,NaN,NaN,NaN,NaN,NaN,2008-09-30,NaN,NaN,NaN
9,14006439,CPG Ventures Limited,0.4430,Medium,0.6000,fixed_score=0.62; ownership_score=0.48; tangib...,0.6957,0.6174,0.4783,0.0,0.3891,840302.0,840302.0,11000000.0,4.0,NaN,2024-06-30,NaN,NaN,NaN


In [33]:
# Save all outputs
print("💾 Saving outputs...")

# Main scored output
final_output.to_csv('phase4_scored_tiers.csv', index=False)
print("   ✓ phase4_scored_tiers.csv")

# Calibration weights
calibration_report = calibrator.get_report()
calibration_report['export_timestamp'] = datetime.now().isoformat()
calibration_report['config'] = {
    'high_tier_threshold': config.high_tier_threshold,
    'medium_tier_threshold': config.medium_tier_threshold
}
calibration_report['data_sources'] = {
    'input_companies': len(companies_df),
    'financials_from_api': len(financials_df),
    'historical_accounts': len(historical_df) if historical_df is not None else 0,
    'land_registry_companies': len(land_registry_df) if land_registry_df is not None else 0
}

with open('phase4_calibrated_weights.json', 'w') as f:
    json.dump(calibration_report, f, indent=2, default=str)
print("   ✓ phase4_calibrated_weights.json")

# Coefficients
if calibrator.coefficients:
    coef_df = pd.DataFrame([
        {'feature': k, 'coefficient': v, 'weight': weights.get(k, 0)}
        for k, v in calibrator.coefficients.items()
    ])
    coef_df.to_csv('phase4_logreg_coefficients.csv', index=False)
    print("   ✓ phase4_logreg_coefficients.csv")

# Tier summary
tier_summary = scored_df.groupby('priority_tier').agg(
    count=('registered_number', 'count'),
    avg_score=('priority_score', 'mean'),
    min_score=('priority_score', 'min'),
    max_score=('priority_score', 'max')
).reset_index()
tier_summary.to_csv('phase4_tier_summary.csv', index=False)
print("   ✓ phase4_tier_summary.csv")

# DueDil API statistics
if DUEDIL_API_KEY and 'duedil_client' in dir():
    api_stats = duedil_client.stats
    api_stats['timestamp'] = datetime.now().isoformat()
    with open('duedil_api_stats.json', 'w') as f:
        json.dump(api_stats, f, indent=2)
    print("   ✓ duedil_api_stats.json")

print("\n✅ All outputs saved!")

💾 Saving outputs...
   ✓ phase4_scored_tiers.csv
   ✓ phase4_calibrated_weights.json
   ✓ phase4_logreg_coefficients.csv
   ✓ phase4_tier_summary.csv
   ✓ duedil_api_stats.json

✅ All outputs saved!


In [34]:
# Download files
from google.colab import files

print("📥 Downloading files...")
files.download('phase4_scored_tiers.csv')
files.download('phase4_calibrated_weights.json')

if calibrator.coefficients:
    files.download('phase4_logreg_coefficients.csv')

files.download('phase4_tier_summary.csv')

print("\n✅ Downloads initiated!")

📥 Downloading files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Downloads initiated!


---
## 📊 Executive Summary

In [35]:
# Generate executive summary
print("="*70)
print("📊 UK CAPITAL ALLOWANCE PRIORITIZATION - EXECUTIVE SUMMARY")
print("="*70)

print(f"\n📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

print(f"\n📥 DATA SOURCES")
print(f"   • Input Companies (CSV): {len(companies_df):,}")
print(f"   • Financials (DueDil API): {len(financials_df):,} retrieved")
if historical_df is not None:
    print(f"   • Historical Accounts (CSV): {len(historical_df):,}")
if land_registry_df is not None:
    print(f"   • Land Registry (CSV): {len(land_registry_df):,} companies")

print(f"\n🎯 TIER DISTRIBUTION")
for tier in ['High', 'Medium', 'Low']:
    count = tier_counts.get(tier, 0)
    pct = count / len(scored_df) * 100
    print(f"   {tier:8s}: {count:,} companies ({pct:.1f}%)")

print(f"\n⚖️ TOP CALIBRATED WEIGHTS")
for feature, weight in sorted(weights.items(), key=lambda x: -x[1])[:3]:
    print(f"   • {feature}: {weight*100:.1f}%")

if calibrator.cv_scores is not None:
    print(f"\n📈 MODEL PERFORMANCE")
    print(f"   Cross-validation AUC: {calibrator.cv_scores.mean():.3f} (±{calibrator.cv_scores.std():.3f})")

print(f"\n📁 OUTPUT FILES")
print(f"   • phase4_scored_tiers.csv - Salesforce-ready prioritized accounts")
print(f"   • phase4_calibrated_weights.json - Model weights for governance")
print(f"   • phase4_logreg_coefficients.csv - Regression coefficients")
print(f"   • phase4_tier_summary.csv - Tier statistics")

print("\n" + "="*70)
print("✅ PRIORITIZATION COMPLETE - Ready for Salesforce Integration")
print("="*70)

📊 UK CAPITAL ALLOWANCE PRIORITIZATION - EXECUTIVE SUMMARY

📅 Analysis Date: 2026-02-05 16:38

📥 DATA SOURCES
   • Input Companies (CSV): 115
   • Financials (DueDil API): 115 retrieved
   • Historical Accounts (CSV): 6,051
   • Land Registry (CSV): 28,280 companies

🎯 TIER DISTRIBUTION
   High    : 22 companies (19.1%)
   Medium  : 36 companies (31.3%)
   Low     : 57 companies (49.6%)

⚖️ TOP CALIBRATED WEIGHTS
   • fixed_score: 33.7%
   • ownership_score: 25.4%
   • ownership_recency_score: 18.2%

📈 MODEL PERFORMANCE
   Cross-validation AUC: 0.605 (±0.117)

📁 OUTPUT FILES
   • phase4_scored_tiers.csv - Salesforce-ready prioritized accounts
   • phase4_calibrated_weights.json - Model weights for governance
   • phase4_logreg_coefficients.csv - Regression coefficients
   • phase4_tier_summary.csv - Tier statistics

✅ PRIORITIZATION COMPLETE - Ready for Salesforce Integration


---

## 📚 Appendix: Data Flow & Methodology

### Data Sources

| Source | Format | Fields Used |
|--------|--------|-------------|
| Input Companies | **CSV** | Registered Number, Account Name |
| Historical Accounts | **CSV** | Registered Number, Signed (0/1) |
| HM Land Registry | **CSV** | Title Number, Tenure, Company Reg No, Date Added |
| DueDil Financials | **API** | tangibleAssets.deltaAbsolute, fixedAssets.deltaAbsolute, numberOfEmployees.value, turnover.value |

### DueDil API Endpoint

```
GET https://duedil.io/v4/company/gb/{company_registration_number}/financials.json
Headers: X-AUTH-TOKEN: {api_key}
```

### Feature Engineering

| Feature | Source | Calculation |
|---------|--------|-------------|
| tangible_score | DueDil API | Percentile rank of tangibleAssets.deltaAbsolute |
| fixed_score | DueDil API | Percentile rank of fixedAssets.deltaAbsolute |
| size_score | DueDil API | Avg(employee percentile, turnover percentile) |
| ownership_score | Land Registry CSV | Property count percentile + freehold bonus |
| ownership_recency_score | Land Registry CSV | Recent acquisition flag (≤3 years) |

### Tier Thresholds

| Tier | Percentile | Description |
|------|------------|-------------|
| High | ≥85th | Top 15% - Priority prospects |
| Medium | 50th-85th | Next 35% - Secondary targets |
| Low | <50th | Bottom 50% - Lower priority |